## Setup :  Time Tracking and Base Tools

In [1]:
import time
import functools
import asyncio
from datetime import datetime
from pprint import pprint

In [2]:
def log_time(fn):
    if asyncio.iscoroutinefunction(fn):
        @functools.wraps(fn)
        async def wrapper(*args, **kwargs):
            start = time.perf_counter()
            print(f"[{fn.__name__}] Started at {datetime.now():%Y-%m-%d %H:%M:%S}")
            result = await fn(*args, **kwargs)
            elapsed = time.perf_counter() - start
            print(f"[{fn.__name__}] Ended at   {datetime.now():%Y-%m-%d %H:%M:%S}")
            print(f"[{fn.__name__}] Elapsed: {elapsed:.4f}s")
            return result
    else:
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            start = time.perf_counter()
            print(f"[{fn.__name__}] Started at {datetime.now():%Y-%m-%d %H:%M:%S}")
            result = fn(*args, **kwargs)
            elapsed = time.perf_counter() - start
            print(f"[{fn.__name__}] Ended at   {datetime.now():%Y-%m-%d %H:%M:%S}")
            print(f"[{fn.__name__}] Elapsed: {elapsed:.4f}s")
            return result
    return wrapper


In [3]:
# Async non-blocking — yields control back to the event loop
@log_time
async def async_wait():
    print("async: start")
    await asyncio.sleep(10)  # non-blocking; other coroutines can run
    print("async: done")

# Sync blocking — holds the thread hostage
@log_time
def sync_wait():
    print("sync: start")
    time.sleep(10)  # blocks the entire thread
    print("sync: done")

## Basic Tools Test

In [20]:
# Base Tools
from agent_base.tools.decorators import tool

@tool
def sync_tool_basic(a: int, b: float):
    """
    This is a basic sync tool that takes two arguments and returns a string.
    Args:
        a: An integer argument
        b: A float argument
    Returns:
        A string "Sync Tool Basic Result"
    """
    sync_wait()
    return "Sync Tool Basic Result"

@tool
async def async_tool_basic(a: list[int], b: set | None = None):
    await async_wait()
    return "Async Tool Basic Result"

In [21]:
from agent_base.sandbox.local import LocalSandbox
from agent_base.tools.registry import ToolRegistry

sandbox = LocalSandbox('s-123', './tool-test-sandbox')

tools = [sync_tool_basic, async_tool_basic]

registry = ToolRegistry()   
registry.attach_sandbox(sandbox)    # Sandbox attachment is optional here if not needed
registry.register_tools(tools)

In [22]:
# Get Schemas
schemas = registry.get_schemas()
pprint(schemas)


[{'description': 'This is a basic sync tool that takes two arguments and '
                 'returns a string.',
  'input_schema': {'properties': {'a': {'description': 'An integer argument',
                                        'type': 'integer'},
                                  'b': {'description': 'A float argument',
                                        'type': 'number'}},
                   'required': ['a', 'b'],
                   'type': 'object'},
  'name': 'sync_tool_basic'},
 {'description': 'Function: async_tool_basic',
  'input_schema': {'properties': {'a': {'description': 'Parameter of type '
                                                       'array',
                                        'items': {'type': 'integer'},
                                        'type': 'array'},
                                  'b': {'description': 'Parameter of type '
                                                       "['object', 'null']",
                                   

In [ ]:
# assume: schemas = registry.get_schemas()
# Schema Export Check.

assert len(schemas) == 2

by_name = {s["name"]: s for s in schemas}
assert set(by_name) == {"sync_tool_basic", "async_tool_basic"}

# sync_tool_basic
sync = by_name["sync_tool_basic"]
assert sync["input_schema"]["required"] == ["a", "b"]
assert sync["input_schema"]["properties"]["a"]["type"] == "integer"
assert sync["input_schema"]["properties"]["b"]["type"] == "number"

# async_tool_basic
async_ = by_name["async_tool_basic"]
assert async_["input_schema"]["required"] == ["a"]  # b should be optional
assert async_["input_schema"]["properties"]["a"]["type"] == "array"
assert async_["input_schema"]["properties"]["a"]["items"]["type"] == "integer"
assert async_["input_schema"]["properties"]["b"]["type"] == ["object", "null"]

In [38]:
from agent_base.tools.registry import ToolCallInfo
tool_call_info = [
    ToolCallInfo(
        name="sync_tool_basic",
        input={"a": 1, "b": 2.0},
        tool_id="sync_tool_basic",
    ),
    ToolCallInfo(
        name="async_tool_basic",
        input={"a": [1, 2, 3]},
        tool_id="async_tool_basic",
    ),
    ToolCallInfo(
        name="async_tool_basic",
        input={"a": [1, 2, 3]},
        tool_id="async_tool_basic",
    ),
    ToolCallInfo(
        name="sync_tool_basic",
        input={"a": 1, "b": 2.0},
        tool_id="sync_tool_basic",
    ),
]

classification = registry.classify_tool_calls(tool_call_info)
pprint(classification)


ToolCallClassification(backend_calls=[ToolCallInfo(name='sync_tool_basic',
                                                   tool_id='sync_tool_basic',
                                                   input={'a': 1, 'b': 2.0}),
                                      ToolCallInfo(name='async_tool_basic',
                                                   tool_id='async_tool_basic',
                                                   input={'a': [1, 2, 3]}),
                                      ToolCallInfo(name='async_tool_basic',
                                                   tool_id='async_tool_basic',
                                                   input={'a': [1, 2, 3]}),
                                      ToolCallInfo(name='sync_tool_basic',
                                                   tool_id='sync_tool_basic',
                                                   input={'a': 1, 'b': 2.0})],
                       frontend_calls=[],
                       confirmati

In [39]:
# Classification Check.
# assume classification = registry.classify_tool_calls(tool_call_info)

assert len(classification.backend_calls) == 4
assert len(classification.frontend_calls) == 0
assert len(classification.confirmation_calls) == 0

assert classification.needs_relay == False

assert classification.backend_calls[0].name == "sync_tool_basic"
assert classification.backend_calls[0].input == {"a": 1, "b": 2.0}
assert classification.backend_calls[0].tool_id == "sync_tool_basic"

In [42]:
start_time = time.monotonic()
tool_results = await registry.execute_tools(classification.backend_calls)
elapsed = time.monotonic() - start_time
print(f"All Tools Elapsed: {elapsed:.4f}s")

[async_wait] Started at 2026-03-01 12:31:07
async: start
[async_wait] Started at 2026-03-01 12:31:07
async: start
[sync_wait] Started at 2026-03-01 12:31:07
sync: start
[sync_wait] Started at 2026-03-01 12:31:07
sync: start
async: done
[async_wait] Ended at   2026-03-01 12:31:17
[async_wait] Elapsed: 10.0011s
async: done
[async_wait] Ended at   2026-03-01 12:31:17
[async_wait] Elapsed: 10.0011s
sync: done
[sync_wait] Ended at   2026-03-01 12:31:17
[sync_wait] Elapsed: 10.0048s
sync: done
[sync_wait] Ended at   2026-03-01 12:31:17
[sync_wait] Elapsed: 10.0048s
All Tools Elapsed: 10.0059s


In [43]:
# Check Parallel Execution from the time spent
assert tool_results[0].duration_ms - 10000 < 1000
assert tool_results[1].duration_ms - 10000 < 1000
assert tool_results[2].duration_ms - 10000 < 1000
assert tool_results[3].duration_ms - 10000 < 1000

assert elapsed - 10000 < 1000

## Configurable Tool Base

In [5]:
from agent_base.sandbox.local import LocalSandbox
from agent_base.tools.registry import ToolRegistry

sandbox = LocalSandbox('s-123', './tool-test-sandbox')

registry = ToolRegistry()   
# registry.attach_sandbox(sandbox)    # Sandbox attachment is optional here if not needed

In [23]:
from agent_base.tools import ConfigurableToolBase
from typing import Dict, Any, Callable

class ConfigurableToolSync(ConfigurableToolBase):
    DOCSTRING_TEMPLATE = """ 
    This is a configurable sync tool that can take parametric descriptions.
    Parameter A : '{a}'
    Parameter B : '{b}'
    """

    def __init__(self, a: int, b: float):
        self.a = a
        self.b = b
        
    def _get_template_context(self) -> Dict[str, Any]:
        return {"a": self.a, "b": self.b}
    
    def get_tool(self) -> Callable:
        instance = self
        @log_time
        def configurable_sync_tool(c: list[list[float]], d: str) -> str:
            '''Optional docstring.'''
            sync_wait()
            return "Configurable Sync Tool Result"
        
        func = self._apply_schema(configurable_sync_tool)
        func.__tool_instance__ = instance
        return func
    
class ConfigurableToolAsync(ConfigurableToolBase):
    DOCSTRING_TEMPLATE = """ 
    This is a configurable async tool that can take parametric descriptions.
    Parameter A : '{a}'
    Parameter B : '{b}'
    """

    def __init__(self, a: int, b: float):
        self.a = a
        self.b = b
        
    def _get_template_context(self) -> Dict[str, Any]:
        return {"a": self.a, "b": self.b}
    
    def get_tool(self) -> Callable:
        instance = self
        @log_time
        async def configurable_async_tool(c: list[list[float]], d: str) -> str:
            await async_wait()
            return "Configurable Async Tool Result"
        
        func = self._apply_schema(configurable_async_tool)
        func.__tool_instance__ = instance
        return func
    
configurable_async_tool = ConfigurableToolAsync(1, 2.0).get_tool()
configurable_sync_tool = ConfigurableToolSync(1, 2.0).get_tool()
tools = [configurable_async_tool, configurable_sync_tool]
registry.register_tools(tools)


In [24]:
# Get Schemas
schemas = registry.get_schemas()
pprint(schemas)

[{'description': 'This is a configurable async tool that can take parametric '
                 'descriptions.\n'
                 "Parameter A : '1'\n"
                 "Parameter B : '2.0'",
  'input_schema': {'properties': {'c': {'description': 'Parameter of type '
                                                       'array',
                                        'items': {'items': {'type': 'number'},
                                                  'type': 'array'},
                                        'type': 'array'},
                                  'd': {'description': 'Parameter of type '
                                                       'string',
                                        'type': 'string'}},
                   'required': ['c', 'd'],
                   'type': 'object'},
  'name': 'configurable_async_tool'},
 {'description': 'This is a configurable sync tool that can take parametric '
                 'descriptions.\n'
                 "Parameter A

In [25]:
# Schema Export Check.

...

Ellipsis

In [26]:
from agent_base.tools.registry import ToolCallInfo
tool_call_info = [
     ToolCallInfo(
        name="configurable_async_tool",
        input={"c": 1, "d": 2.0},
        tool_id="configurable_async_tool",
    ),
    ToolCallInfo(
        name="configurable_sync_tool",
        input={"c": 1, "d": 2.0},
        tool_id="configurable_sync_tool",
    ),
    ToolCallInfo(
        name="configurable_async_tool",
        input={"c": 1, "d": 2.0},
        tool_id="configurable_async_tool",
    ),
    ToolCallInfo(
        name="configurable_sync_tool",
        input={"c": 1, "d": 2.0},
        tool_id="configurable_sync_tool",
    ),
]


In [27]:
classification = registry.classify_tool_calls(tool_call_info)
pprint(classification)

ToolCallClassification(backend_calls=[ToolCallInfo(name='configurable_async_tool',
                                                   tool_id='configurable_async_tool',
                                                   input={'c': 1, 'd': 2.0}),
                                      ToolCallInfo(name='configurable_sync_tool',
                                                   tool_id='configurable_sync_tool',
                                                   input={'c': 1, 'd': 2.0}),
                                      ToolCallInfo(name='configurable_async_tool',
                                                   tool_id='configurable_async_tool',
                                                   input={'c': 1, 'd': 2.0}),
                                      ToolCallInfo(name='configurable_sync_tool',
                                                   tool_id='configurable_sync_tool',
                                                   input={'c': 1, 'd': 2.0})],
               

In [28]:
start_time = time.monotonic()
tool_results = await registry.execute_tools(classification.backend_calls)
elapsed = time.monotonic() - start_time
print(f"All Tools Elapsed: {elapsed:.4f}s")
# Check Parallel Execution from the time spent
assert tool_results[0].duration_ms - 10000 < 1000
assert tool_results[1].duration_ms - 10000 < 1000
assert tool_results[2].duration_ms - 10000 < 1000
assert tool_results[3].duration_ms - 10000 < 1000

assert elapsed - 10000 < 1000

[configurable_async_tool] Started at 2026-03-01 12:59:37
[async_wait] Started at 2026-03-01 12:59:37
async: start
[configurable_async_tool] Started at 2026-03-01 12:59:37
[async_wait] Started at 2026-03-01 12:59:37
async: start
[configurable_sync_tool] Started at 2026-03-01 12:59:37
[sync_wait] Started at 2026-03-01 12:59:37
sync: start
[configurable_sync_tool] Started at 2026-03-01 12:59:37
[sync_wait] Started at 2026-03-01 12:59:37
sync: start
sync: donesync: done
[sync_wait] Ended at   2026-03-01 12:59:47
[sync_wait] Elapsed: 10.0010s
[configurable_sync_tool] Ended at   2026-03-01 12:59:47
[configurable_sync_tool] Elapsed: 10.0010s
async: done
[async_wait] Ended at   2026-03-01 12:59:47
[async_wait] Elapsed: 10.0014s
[configurable_async_tool] Ended at   2026-03-01 12:59:47
[configurable_async_tool] Elapsed: 10.0014s
async: done
[async_wait] Ended at   2026-03-01 12:59:47
[async_wait] Elapsed: 10.0013s
[configurable_async_tool] Ended at   2026-03-01 12:59:47
[configurable_async_tool]